Data Ingestion with Lakeflow Connect
This course provides a comprehensive introduction to Lakeflow Connect as a scalable and simplified solution for ingesting data into Databricks from a variety of data sources. You will begin by exploring the different types of connectors within Lakeflow Connect (Standard and Managed), learn about various ingestion techniques, including batch, incremental batch, and streaming, and then review the key benefits of Delta tables and the Medallion architecture.

From there, you will gain practical skills to efficiently ingest data from cloud object storage using Lakeflow Connect Standard Connectors with methods such as CREATE TABLE AS (CTAS), COPY INTO, and Auto Loader, along with the benefits and considerations of each approach. You will then learn how to append metadata columns to your bronze level tables during ingestion into the Databricks data intelligence platform. This is followed by working with the rescued data column, which handles records that don’t match the schema of your bronze table, including strategies for managing this rescued data.

The course also introduces techniques for ingesting and flattening semi-structured JSON data, as well as enterprise-grade data ingestion using Lakeflow Connect Managed Connectors.

Finally, learners will explore alternative ingestion strategies, including MERGE INTO operations and leveraging the Databricks Marketplace, equipping you with foundational knowledge to support modern data engineering ingestion.

Prerequisites
You should meet the following prerequisites before starting this course:

Basic understanding of the Databricks Data Intelligence platform, including Databricks Workspaces, Apache Spark, Delta Lake, the Medallion Architecture and Unity Catalog.
Experience working with various file formats (e.g., Parquet, CSV, JSON, TXT).
Proficiency in SQL and Python.
Familiarity with running code in Databricks notebooks.
Course Agenda
The following modules are part of the Data Engineer Learning Path by Databricks Academy.

#	Notebook Name
00	Appendix
01	Exploring the Lab Environment
02A	Data Ingestion with CREATE TABLE AS and COPY INTO
02B	Create Streaming Tables with SQL using Auto Loader
03	Adding Metadata Columns During Ingestion
04	Handling CSV Ingestion with the Rescued Data Column
05L	Creating Bronze Tables from CSV Files
06	Ingesting JSON files with Databricks
07L	Creating Bronze Tables from JSON Files
08	Enterprise Data Ingestion with Lakeflow Connect
09	BONUS - Data Ingestion with MERGE INTO
Requirements
Please review the following requirements before starting the lesson:

To run demo and lab notebooks, you need to use the following Databricks runtime: 17.3.x-scala2.13
© 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the Apache Software Foundation.

Privacy Policy | Terms of Use | Support

1 - Exploring the Lab Environment
This demonstration is meant as a review for understanding data objects registered to Unity Catalog (UC) using Databricks. UC is a unified data governance solution designed to centralize and streamline the management of data, metadata, and access control across multiple Databricks workspaces. It provides interoperability across lakehouse formats like Delta lake and Apache Iceberg in addition to providing open APIs and built-in governance for data and AI applications.

Learning Objectives
By the end of this lesson, you should be able to:

Identify and display available Unity Catalog objects, including catalogs, schemas, volumes, and tables within a Databricks.
Execute SQL queries to display data directly from files in cloud storage.
References For more additional reading and learning, check out the official UC GitHub repository and this video on UC on Databricks.

REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that Serverless is enabled by default and you have a Shared SQL warehouse.

Follow these steps to select the classic compute cluster:

Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use Serverless.

If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

Click More in the drop-down.

In the Attach to an existing compute resource window, use the first drop-down to select your unique cluster.

NOTE: If your cluster has terminated, you might need to restart it in order to select it. To do this:

Right-click on Compute in the left navigation pane and select Open in new tab.

Find the triangle icon to the right of your compute cluster name and click it.

Wait a few minutes for the cluster to start.

Once the cluster is running, complete the steps above to select your cluster.

A. Classroom Setup
Run the following cell to configure your working environment for this notebook.
NOTE: The DA object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course in the lab environment.



In [0]:
%sql
%run ./Includes/Classroom-Setup-01

2 . Complete the following to explore your labuser schema using the Catalog UI on the left.

a. In the left navigation bar, select the catalog icon: Catalog Icon

b. Locate the catalog called dbacademy and expand the catalog.

c. Expand the labuser schema (database). This is your schema for the course. It should be your lab username (for example, labuser1234_5678).


3.We want to modify our default catalog and default schema to use dbacademy and our labuser schema to avoid writing the three-level namespace every time we query and create tables in this course.

However, before we proceed, note that each of us has a different schema name. Your specific schema name has been stored dynamically in the SQL variable DA.schema_name during the classroom setup script.

Run the code below and confirm that the value of the DA.schema_name SQL variable matches your specific schema name (e.g., labuser1234_678).

In [0]:
%sql
values(DA.schema_name)

Let's modify our default catalog and schema using the USE CATALOG and USE SCHEMA statements. This eliminates the need to specify the three-level name for objects in your labuser schema (i.e., catalog.schema.object).

USE CATALOG – Sets the current catalog.

USE SCHEMA – Sets the current schema.

NOTE: Since our dynamic schema name is stored in the SQL variable DA.schema_name as a string, we will need to use the IDENTIFIER clause to interpret the constant string in our variable as a schema name. The IDENTIFIER clause can interpret a constant string as any of the following:

Relation (table or view) name
Function name
Column name
Field name
Schema name
Catalog name
IDENTIFIER clause documentation

Run the following cell to set and view your default catalog and schema. Confirm that your default catalog is dbacademy and your schema is labuser (this uses the DA.schema_name variable created in the classroom setup script).

NOTE: Alternatively, you can simply add your schema name without using the IDENTIFIER clause.

In [0]:
%sql
-- Change the default catalog/schema
USE CATALOG dbacademy;
USE SCHEMA IDENTIFIER(DA.schema_name);


-- View current catalog and schema
SELECT 
  current_catalog(), 
  current_schema()

B. Inspecting and Referencing Unity Catalog Objects
Catalogs, Schemas, Volumes, and Tables
In Unity Catalog, all metadata is registered in a metastore. The hierarchy of database objects in any Unity Catalog metastore is divided into three levels, represented as a three-level namespace (example, <catalog>.<schema>.<object>) when you reference tables, views, volumes, models, and functions.

B1. Catalogs
Use the SHOW SCHEMAS IN statement to view available schemas in the dbacademy catalog. Run the cell and view the results. Notice that your labuser schema is within the dbacademy catalog.

In [0]:
%sql
SHOW SCHEMAS IN dbacademy;

B2. Schemas
Run the DESCRIBE SCHEMA EXTENDED statement to see information about your labuser schema (database) that was created for you within the dbacademy catalog. In the output below, your schema name is in the row called Namespace Name.

NOTE: Remember, we are using the IDENTIFIER clause to dynamically reference your specific schema name in the lab, since each user will have a different schema name. Alternatively, you can type in the schema name.

In [0]:
%sql
DESCRIBE SCHEMA EXTENDED IDENTIFIER(DA.schema_name);


B3. Tables
Use the DESCRIBE TABLE EXTENDED statement to describe the table mydeltatable.

Run the cell and view the results. Notice the following:

In the first few cells, you can see column information.
Starting at cell 4, you can see additional Delta Statistics Columns.
Starting at cell 8, you can see additional Detailed Table Information.
NOTE: Remember, we do not need to reference the three-level namespace (catalog.schema.table) because we set our default catalog and schema earlier.

In [0]:
%sql
DESCRIBE TABLE EXTENDED mydeltatable

B4. Volumes
Volumes are Unity Catalog objects that enable governance over non-tabular datasets. Volumes represent a logical volume of storage in a cloud object storage location. Volumes provide capabilities for accessing, storing, governing, and organizing files.

While tables provide governance over tabular datasets, volumes add governance over non-tabular datasets. You can use volumes to store and access files in any format, including structured, semi-structured, and unstructured data.

Databricks recommends using volumes to govern access to all non-tabular data. Like tables, volumes can be managed or external.

B4.1 UI Exploration
Complete the following to explore the dbacademy_ecommerce catalog:

1.In the left navigation bar, select the catalog icon: Catalog Icon

2.Locate the catalog called dbacademy_ecommerce and expand the catalog.

3.Expand the v01 schema. Notice that this catalog contains two volumes, delta and raw.

4.Expand the raw volume. Notice that the volume contains a series of folders.

5. Expand the users-historical folder. Notice that the folder contains a series of files.



B4.2 Volume Exploration with SQL
Run the DESCRIBE VOLUME statement to return the metadata for the dbacademy_ecommerce.v01.raw volume. The metadata includes the volume name, schema, catalog, type, comment, owner, and more.

Notice the following:

Under the storage_location column, you can see the cloud storage location for this volume.

Under the volume_type column, it indicates this is a MANAGED volume.

In [0]:
%sql
DESCRIBE VOLUME dbacademy_ecommerce.v01.raw;

RESULT:name	catalog	database	owner	storage_location	volume_type	comment	securable_type	securable_kind
raw	dbacademy_ecommerce	v01	metastore_admins	s3://marketplace-sandbox-uc-databricks/UC/8f245c11-89d6-49ba-b4cb-8698c92dc4fe/volumes/01312a33-a599-4c00-ab89-9702aff680e1	MANAGED	null	VOLUME	VOLUME_DELTASHARING

B4.3 List Files in a Volume
Use the LIST statement to list the available files in the raw volume's users-historical directory (/Volumes/dbacademy_ecommerce/v01/raw/users-historical) and view the results.

Notice the following:

Ignore any file names that begin with an underscore (_). These are temporary or intermediate files used when writing files to a location.
Scroll down in the results and expand one of the files where the name column begins with part. Confirm that this directory contains a series of Parquet files.
NOTE: When interacting with data in volumes, use the path provided by Unity Catalog, which always follows this format: /Volumes/catalog_name/schema_name/volume_name/.

For more information on exploring directories and data files managed with Unity Catalog volumes, check out the Explore storage and find data files documentation.

In [0]:
%sql
LIST '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'

RESULT:
path	name	size	modification_time
/Volumes/dbacademy_ecommerce/v01/raw/users-historical/_SUCCESS	_SUCCESS	0	1726173048000
/Volumes/dbacademy_ecommerce/v01/raw/users-historical/_committed_531959640415905750	_committed_531959640415905750	424	1726173048000
/Volumes/dbacademy_ecommerce/v01/raw/users-historical/_started_531959640415905750	_started_531959640415905750	0	1726173049000
/Volumes/dbacademy_ecommerce/v01/raw/users-historical/part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet	part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet	974753	1726173049000
/Volumes/dbacademy_ecommerce/v01/raw/users-historical/part-00001-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7572-1-c000.snappy.parquet	part-00001-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7572-1-c000.snappy.parquet	976470	1726173049000
/Volumes/dbacademy_ecommerce/v01/raw/users-historical/part-00002-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7573-1-c000.snappy.parquet	part-00002-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7573-1-c000.snappy.parquet	980390	1726173049000
/Volumes/dbacademy_ecommerce/v01/raw/users-historical/part-00003-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7574-1-c000.snappy.parquet	part-00003-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7574-1-c000.snappy.parquet	979632	1726173050000

### 2A - Data Ingestion with CREATE TABLE AS and COPY INTO ###

2A - Data Ingestion with CREATE TABLE AS and COPY INTO
In this demonstration, we'll explore ingesting data from cloud storage into Delta tables with the CREATE TABLE AS (CTAS) AND COPY INTO statements.

Learning Objectives
By the end of this lesson, you should be able to:

Use the CTAS statement with read_files() to ingest Parquet files into a Delta table.
Use COPY INTO to incrementally load Parquet files from cloud object storage into a Delta table.
REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that Serverless is enabled by default and you have a Shared SQL warehouse.

Follow these steps to select the classic compute cluster:

Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use Serverless.

If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

Click More in the drop-down.

In the Attach to an existing compute resource window, use the first drop-down to select your unique cluster.

NOTE: If your cluster has terminated, you might need to restart it in order to select it. To do this:

Right-click on Compute in the left navigation pane and select Open in new tab.

Find the triangle icon to the right of your compute cluster name and click it.

Wait a few minutes for the cluster to start.

Once the cluster is running, complete the steps above to select your cluster.

A. Classroom Setup
Run the following cell to configure your working environment for this notebook.

NOTE: The DA object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course in the lab environment.



In [0]:
%sql
%run ./Includes/Classroom-Setup-02

Run the cell below to view your default catalog and schema. Notice that your default catalog is dbacademy and your default schema is your unique labuser schema.

NOTE: The default catalog and schema are pre-configured for you to avoid the need to specify the three-level name when writing your tables to your labuser schema (i.e., catalog.schema.table).

In [0]:
%SQL
SELECT current_catalog(), current_schema()

B. Explore the Data Source Files
We'll create a table containing historical user data from Parquet files stored in the volume
'/Volumes/dbacademy_ecommerce/v01/raw/users-historical' within Unity Catalog.

Use the LIST statement to view the files in this volume. Run the cell and review the results.

Notice the files in the name column begin with part-. This shows that this volume contains multiple Parquet files.

In [0]:
%sql
--List files in a raw/users-historical volume
LIST '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'

Query the Parquet files by path in the /Volumes/dbacademy_ecommerce/v01/raw/users-historical directory to view the raw data in tabular format to quickly preview the files.

In [0]:
%sql
SELECT * FROM json.`/Volumes/catalog_name/schema_name/volume_name/path/to/data`

In [0]:
spark.read.format("json").load("/Volumes/catalog_name/schema_name/volume_name/path/to/data")

In [0]:
%sql
---view parquet data
SELECT * 
FROM parquet.`/Volumes/dbacademy_ecommerce/v01/raw/users-historical`;

C. Batch Data Ingestion with CTAS and read_files()
The CREATE TABLE AS (CTAS) statement is used to create and populate tables using the results of a query. This allows you to create a table and load it with data in a single step, streamlining data ingestion workflows.

Automatic Schema Inference for Parquet Files
Apache Parquet is a columnar storage format optimized for analytical queries. It includes embedded schema metadata (e.g., column names and data types), which enables automatic schema inference when creating tables from Parquet files. This eliminates the need for manual schema definitions and simplifies the process of converting Parquet files into Delta format by leveraging the built-in schema metadata.


C1. CTAS with the read_files() Function
The code in the next cell creates a table using CTAS with the read_files() function.

The read_files() table-valued function (TVF) enables reading a variety of file formats and provides additional options for data ingestion.

1.First, let's explore the documentation for read_files. Complete the following steps:

a. Navigate to the read_files documentation.

b. Scroll down and find the Options section. Take a moment to explore some of the features of read_files.

c. In the Options section, notice the variety of options available based on the file type.

d. Click on parquet and scroll through the available options.

NOTE: The read_files function provides a wide range of capabilities and specific options for each file type. The previous method used to create a table only works if no additional options are required.

2.Use the read_files() function to query the same Parquet files located in /Volumes/dbacademy_ecommerce/v01/raw/users-historical. The LIMIT clause limits the amount of rows during exploration and development.

The first parameter in read_files is the path to the data.

The format => "parquet" option specifies the file format.

The read_files function automatically detects the file format and infers a unified schema across all files. It also supports explicit schema definitions and schemaHints. For more details on schema inference capabilities, refer to the Schema inference documentation.

NOTE: A _rescued_data column is automatically included by default to capture any data that doesn’t match the inferred schema.

In [0]:
%sql
----17:Read parquet with read_files
SELECT * 
FROM read_files(
  '/Volumes/dbacademy_ecommerce/v01/raw/users-historical',
  format => 'parquet'
)
LIMIT 10;

3.Next, let's use read_files() with a CTAS statement to create the table historical_users_bronze_ctas_rf, then display the table.

Notice that the Parquet files were ingested to create a table (Delta by default).

In [0]:
%sql
----19:Create table with read_files
-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS historical_users_bronze_ctas_rf;


-- Create the Delta table
CREATE TABLE historical_users_bronze_ctas_rf 
SELECT * 
FROM read_files(
        '/Volumes/dbacademy_ecommerce/v01/raw/users-historical',
        format => 'parquet'
      );


-- Preview the Delta table
SELECT * 
FROM historical_users_bronze_ctas_rf 
LIMIT 10;

4.Run the DESCRIBE TABLE EXTENDED statement to view column names, data types, and additional table metadata.

Review the results and notice the following:

The table was created in your schema within the course catalog dbacademy.

The Type row indicates that the table is MANAGED.

The Location row shows the managed cloud storage location.

The Provider row specifies that the table is a Delta table.

In [0]:
DESCRIBE TABLE EXTENDED historical_users_bronze_ctas_rf;

Managed vs External Tables in Databricks
Managed Tables
Databricks manages both the data and metadata.
Data is stored within Databricks’ managed storage.
Dropping the table also deletes the data.
Recommended for creating new tables.
External Tables
Databricks only manages the table metadata.
Dropping the table does not delete the data.
Supports multiple formats, including Delta Lake.
Ideal for sharing data across platforms or using existing external data.

C2. (BONUS) Python Ingestion
The code uses Python to ingest the parquet files.



In [0]:
%python
# 1. Read the Parquet files from the volume into a Spark DataFrame
df = (spark
      .read
      .format("parquet")
      .load("/Volumes/dbacademy_ecommerce/v01/raw/users-historical")
    )


# 2. Write to the DataFrame to a Delta table (overwrite the table if it exists)
(df
 .write
 .mode("overwrite")
 .saveAsTable(f"dbacademy.{DA.schema_name}.historical_users_bronze_python")
)


## 3. Read and view the table
users_bronze_table = spark.table(f"dbacademy.{DA.schema_name}.historical_users_bronze_python")
users_bronze_table.display()


D. Incremental Data Ingestion with COPY INTO
COPY INTO is a Databricks SQL command that allows you to load data from a file location into a Delta table. This operation is re-triable and idempotent, i.e. files in the source location that have already been loaded are skipped. This command is useful for when you need to load data into an existing Delta table.

COPY INTO



D1. Ingesting Parquet Files with COPY INTO
Using the same set of Parquet files as before, let's use COPY INTO to create our Bronze table again.

We will look at two examples:

Example 1: Common Schema Mismatch Error

Example 2: Preemptively Handling Schema Evolution


Example 1: Common Schema Mismatch Error
The cell below creates an empty table named historical_users_bronze_ci with a defined schema for only the user_id and user_first_touch_timestamp columns.

However, the Parquet files in '/Volumes/dbacademy_ecommerce/v01/raw/users-historical' contain three columns:

user_id
user_first_touch_timestamp
email
Run the cell below and review the error. You should see the [COPY_INTO_SCHEMA_MISMATCH_WITH_TARGET_TABLE] error. This error occurs because there is a schema mismatch: the Parquet files contain 3 columns, but the target table historical_users_bronze_ci only has 2 columns.

How can you handle this error?

In [0]:
%sql
----COPY INTO schema mismatch

--------------------------------------------
-- This cell returns an error
--------------------------------------------

-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS historical_users_bronze_ci;


-- Create an empty table with the specified table schema (only 2 out of the 3 columns)
CREATE TABLE historical_users_bronze_ci (
  user_id STRING,
  user_first_touch_timestamp BIGINT
);


-- Use COPY INTO to populate Delta table
COPY INTO historical_users_bronze_ci
  FROM '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'
  FILEFORMAT = parquet;

In [0]:
2.We can fix this error by adding COPY_OPTIONS with the mergeSchema = 'true' option. When set to true, this option allows the schema to evolve based on the incoming data.

Run the next cell with the COPY_OPTIONS option added. You should notice that the Parquet files were successfully ingested into the table, with a total of 251,501 rows ingested.

In [0]:
COPY INTO historical_users_bronze_ci
  FROM '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'
  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');     -- Merge the schema of each file

Preview the data in the historical_users_bronze_ci table.

In [0]:
%sql
SELECT *
FROM historical_users_bronze_ci
LIMIT 10;

Example 2: Preemptively Handling Schema Evolution
Another way to ingest the same files into a Delta table is to start by creating an empty table named historical_users_bronze_ci_no_schema.

Then, add the COPY_OPTIONS ('mergeSchema' = 'true') option to enable schema evolution for the table.

Run the cell and confirm that 251,501 rows were added to the Delta table.

In [0]:
%sql
---COPY INTO with schema evolution---
-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS historical_users_bronze_ci_no_schema;


-- Create an empty table without the specified schema
CREATE TABLE historical_users_bronze_ci_no_schema;


-- Use COPY INTO to populate Delta table
COPY INTO historical_users_bronze_ci_no_schema
  FROM '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'
  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');

D2. Idempotency (Incremental Ingestion)
COPY INTO tracks the files it has previously ingested. If the command is run again, no additional data is ingested because the files in the source directory haven't changed.

1.Let's run the COPY INTO command again and check if any data is re-ingested into the table.

Run the cell and view the results. Notice that the values for num_affected_rows, num_inserted_rows, and num_skipped_corrupt_files are all 0 because the data has already been ingested into the Delta table.

NOTE: If new files are added to the cloud storage location, COPY INTO will only ingest those files. Using COPY INTO is a great option if you want to run a job for incremental batch ingestion from cloud storage location without re-reading files that have already been loaded.

In [0]:
%sql
---COPY INTO idempotency

COPY INTO historical_users_bronze_ci_no_schema
  FROM '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'
  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');

NOTE: For the remaining demos, we will not focus on COPY INTO. Instead, we will focus on leveraging the read_files function. As you gain experience you can begin to create streaming tables for incremental or streaming ingestion using SQL and DLT.